In [13]:
import os
import json
import zipfile
import numpy as np
import copy
import requests
import io
import shutil

import pandas as pd
import geopandas as gpd

In [ ]:
DIR="/home/dy23a.fsu/st/datasets/raw"
os.makedirs(DIR, exist_ok=True)
DIR=os.path.join(DIR,"SF")
os.makedirs(DIR, exist_ok=True)

# SF Open Data (Socrata) does not require an app token for these volumes, but
# leave the credentials here in case rate-limiting kicks in.
SF_Data_Portal_KEY_ID = ""
SF_Data_Portal_KEY_SECRET = ""

sf_shape="./SF Analysis Neighborhoods.geojson"
sf_shapefile_url="https://data.sfgov.org/api/v3/views/j2bu-swwd/query.geojson"

taxi_orig_file=os.path.join(DIR,"sf_taxi_2023.csv")
bike_orig_dir=os.path.join(DIR,"bike")

# Download
Analysis Neighborhoods https://data.sfgov.org/Geographic-Locations-and-Boundaries/Analysis-Neighborhoods/j2bu-swwd
Taxi 2023 https://data.sfgov.org/Transportation/SFMTA-Taxi-Trips-2024-/m8hk-2ipk
Bike 2023 https://s3.amazonaws.com/baywheels-data/index.html

**SF specifics** — only **two** mobilities (taxi + bike); there is no
TNP/scooter feed analogous to Chicago. Both feeds are **GPS-based** (lat/lon),
unlike Chicago's community-area IDs, so the flow/OD notebooks spatially join
every endpoint to an analysis-neighborhood polygon. 

## 1 Shapefile

In [15]:
if not os.path.exists(sf_shape):
    response = requests.get(sf_shapefile_url)
    with open(sf_shape, "w", encoding="utf-8") as file:
        file.write(response.text)

## 2 Taxi

The SFMTA Taxi Trips 2023 dataset (`m8hk-2ipk`, ~1.47M rows for 2023) is pulled
from the Socrata SODA endpoint with the same keyset-paginated downloader used in
the Chicago notebook. SF rows have no `trip_id`, so we page by `start_time_local`
and break ties on the Socrata system column `:id`. A `$select` keeps only the
columns the flow / OD notebooks need (timestamps + pickup/dropoff lat/lon).

In [16]:
import csv
import time
import random

from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except ImportError:
    from requests.packages.urllib3.util.retry import Retry

def _last_nonempty_line(path, tail_bytes=64 * 1024):
    with open(path, "rb") as fh:
        fh.seek(0, 2)
        size = fh.tell()
        if size == 0:
            return None
        fh.seek(max(0, size - tail_bytes))
        chunks = fh.read().split(b"\n")
        while chunks and not chunks[-1].strip():
            chunks.pop()
        return chunks[-1].decode("utf-8") if chunks else None

def _count_data_rows(path):
    with open(path, "rb") as fh:
        return max(0, sum(1 for _ in fh) - 1)

def _make_session():
    """Session with transport-level retry for DNS / connect failures."""
    session = requests.Session()
    if SF_Data_Portal_KEY_ID or SF_Data_Portal_KEY_SECRET:
        session.auth = (SF_Data_Portal_KEY_ID, SF_Data_Portal_KEY_SECRET)
    retry = Retry(
        total=10,
        connect=10,
        read=0,            # read retries handled in outer loop with page-size shrink
        status=5,
        backoff_factor=2.0,        # 2, 4, 8, 16, ... seconds
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=32, pool_maxsize=32)
    session.mount("http://", adapter)
    session.mount("https://", adapter)
    return session

def download_from_api(API_URL, base_where, file, order_col, tie_col=None,
                      select_cols=None, page_size=20000, max_retries=50,
                      request_timeout=120, min_page_size=2000):
    """Keyset-paginated downloader for Socrata SODA endpoints.

    Avoids deep $offset (which goes O(offset) on the server and times out beyond
    a few million rows). Resumes by reading the last row of `file` and using
    its (order_col, tie_col) as a cursor.

    `select_cols`, when given, is the value of the SODA `$select` clause (a comma
    separated column list); it MUST include order_col and tie_col so the cursor
    can be read back from the file on resume.

    On repeated ReadTimeouts the page size is halved (down to `min_page_size`)
    so a struggling endpoint can still make progress; it ramps back up after a
    successful page. DNS / connection failures use exponential backoff with
    jitter to ride out resolver flaps.
    """
    cursor_order = None
    cursor_tie = None
    header_cols = None
    header_written = False
    total = 0
    mode = "w"

    if os.path.exists(file) and os.path.getsize(file) > 0:
        with open(file, "r", encoding="utf-8") as fh:
            header_line = fh.readline().rstrip("\n")
        if header_line:
            header_cols = next(csv.reader([header_line]))
            last = _last_nonempty_line(file)
            if last and last != header_line:
                last_row = next(csv.reader([last]))
                row_dict = dict(zip(header_cols, last_row))
                cursor_order = row_dict.get(order_col)
                if tie_col:
                    cursor_tie = row_dict.get(tie_col)
                total = _count_data_rows(file)
                header_written = True
                mode = "a"
                tie_str = f", {tie_col}>'{cursor_tie}'" if cursor_tie else ""
                print(f"Resuming after {order_col}='{cursor_order}'{tie_str} "
                      f"({total} existing rows)")

    order_clause = f"{order_col} ASC" + (f", {tie_col} ASC" if tie_col else "")

    session = _make_session()

    cur_page_size = page_size

    with open(file, mode=mode, encoding="utf-8") as f:
        while True:
            if cursor_order is None:
                where = base_where
            elif tie_col and cursor_tie is not None:
                where = (
                    f"({base_where}) AND "
                    f"({order_col} > '{cursor_order}' OR "
                    f"({order_col} = '{cursor_order}' AND {tie_col} > '{cursor_tie}'))"
                )
            else:
                where = f"({base_where}) AND {order_col} > '{cursor_order}'"

            query = {"$where": where, "$order": order_clause, "$limit": cur_page_size}
            if select_cols:
                query["$select"] = select_cols

            response = None
            for attempt in range(1, max_retries + 1):
                try:
                    response = session.get(API_URL, params=query, timeout=request_timeout)
                    break
                except requests.exceptions.RequestException as e:
                    is_timeout = isinstance(e, (requests.exceptions.ReadTimeout,
                                                requests.exceptions.ConnectTimeout))
                    is_conn = isinstance(e, requests.exceptions.ConnectionError)
                    msg = str(e)
                    is_dns = ("Name or service not known" in msg
                              or "Temporary failure in name resolution" in msg
                              or "nodename nor servname" in msg
                              or "getaddrinfo failed" in msg)

                    if is_timeout and cur_page_size > min_page_size:
                        cur_page_size = max(min_page_size, cur_page_size // 2)
                        query["$limit"] = cur_page_size
                        print(f"Attempt {attempt} timed out: {e}. "
                              f"Reducing page_size to {cur_page_size} ...")
                    if attempt == max_retries:
                        print(f"Failed after {max_retries} attempts: {e}")
                        raise

                    if is_dns or is_conn:
                        # DNS / connect failure: longer exponential backoff + jitter
                        # so concurrent threads don't retry in lockstep.
                        wait = min(300, (2 ** min(attempt, 8))) + random.uniform(0, 5)
                        kind = "DNS" if is_dns else "connection"
                        print(f"Attempt {attempt} {kind} error: {e}. "
                              f"Retrying in {wait:.1f}s ...")
                    else:
                        wait = min(120, 5 * attempt)
                        print(f"Attempt {attempt} failed: {e}. Retrying in {wait}s ...")
                    time.sleep(wait)

            if response.status_code != 200:
                print(f"Error: {response.status_code}")
                print(f"Info: {response.text}")
                return

            lines = response.text.splitlines()
            if len(lines) <= 1:
                break

            header_resp, data_lines = lines[0], lines[1:]
            if not header_written:
                f.write(header_resp + "\n")
                header_cols = next(csv.reader([header_resp]))
                header_written = True
            elif header_cols is None:
                header_cols = next(csv.reader([header_resp]))

            for line in data_lines:
                f.write(line + "\n")
            f.flush()

            last_row = next(csv.reader([data_lines[-1]]))
            row_dict = dict(zip(header_cols, last_row))
            cursor_order = row_dict.get(order_col)
            if tie_col:
                cursor_tie = row_dict.get(tie_col)

            n = len(data_lines)
            total += n
            print(f"Downloaded {total} (page_size={cur_page_size}) ...")

            if cur_page_size < page_size:
                cur_page_size = min(page_size, cur_page_size * 2)

            if n < query["$limit"]:
                break

    print(f"\nSaved {total}")

In [ ]:
DOWNLOAD=False
if DOWNLOAD:
    API_URL = "https://data.sfgov.org/resource/m8hk-2ipk.csv"
    base_where = (
        "start_time_local >= '2023-01-01T00:00:00.000' "
        "AND start_time_local < '2024-01-01T00:00:00.000'"
    )
    # SF taxi rows have no trip_id; break ties on the Socrata system column :id.
    # $select keeps only what the flow / OD notebooks consume (and must include
    # the order + tie columns so a resume can recover the cursor).
    select_cols = (
        ":id, start_time_local, end_time_local, "
        "pickup_location_latitude, pickup_location_longitude, "
        "dropoff_location_latitude, dropoff_location_longitude"
    )
    download_from_api(
        API_URL, base_where, taxi_orig_file,
        order_col="start_time_local", tie_col=":id",
        select_cols=select_cols,
    )

Downloaded 20000 (page_size=20000) ...
Downloaded 40000 (page_size=20000) ...
Downloaded 60000 (page_size=20000) ...
Downloaded 80000 (page_size=20000) ...
Downloaded 100000 (page_size=20000) ...
Downloaded 120000 (page_size=20000) ...
Downloaded 140000 (page_size=20000) ...
Downloaded 160000 (page_size=20000) ...
Downloaded 180000 (page_size=20000) ...
Downloaded 200000 (page_size=20000) ...
Downloaded 220000 (page_size=20000) ...
Downloaded 240000 (page_size=20000) ...
Downloaded 260000 (page_size=20000) ...
Downloaded 280000 (page_size=20000) ...
Downloaded 300000 (page_size=20000) ...
Downloaded 320000 (page_size=20000) ...
Downloaded 340000 (page_size=20000) ...
Downloaded 360000 (page_size=20000) ...
Downloaded 380000 (page_size=20000) ...
Downloaded 400000 (page_size=20000) ...
Downloaded 420000 (page_size=20000) ...
Downloaded 440000 (page_size=20000) ...
Downloaded 460000 (page_size=20000) ...
Downloaded 480000 (page_size=20000) ...
Downloaded 500000 (page_size=20000) ...
Down

## 3 Bike

Bay Wheels (Lyft) publishes monthly trip files on S3 as
`{YYYYMM}-baywheels-tripdata.csv.zip`. Each archive holds a single CSV with
station lat/lon (`start_lat/lng`, `end_lat/lng`) and ISO timestamps
(`started_at` / `ended_at`), the same schema as Citi Bike / Divvy.

In [ ]:
DOWNLOAD=False
year = 2023
base_url_template = "https://s3.amazonaws.com/baywheels-data/{year}{month:02d}-baywheels-tripdata.csv.zip"
os.makedirs(bike_orig_dir, exist_ok=True)
if DOWNLOAD:
    for month in range(1, 13):
        url = base_url_template.format(year=year, month=month)
        zip_filename = f"{year}{month:02d}-baywheels-tripdata.csv.zip"
        zip_filepath = os.path.join(bike_orig_dir, zip_filename)
        try:
            response = requests.get(url, stream=True, timeout=120)

            if response.status_code == 200:
                with open(zip_filepath, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192 * 1024):  # 8MB
                        if chunk:
                            f.write(chunk)

                with zipfile.ZipFile(zip_filepath, 'r') as zip_ref:
                    zip_ref.extractall(bike_orig_dir)
                os.remove(zip_filepath)

        except requests.exceptions.RequestException as e:
            print(f"Error: {e}")